In [1]:
import requests
import pandas as pd

# Cargo API endpoint for structured Dustloop data
api_url = "https://www.dustloop.com/wiki/api.php"

# 'input' is required to catch names like 5P, 2K, etc.
params = {
    "action": "cargoquery",
    "tables": "MoveData_GGST",
    "fields": "name, input, damage, guard, startup, active, recovery, onBlock, invuln",
    "where": 'chara="Sol Badguy"',
    "format": "json",
    "limit": "500" 
}

headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(api_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    raw_results = data.get('cargoquery', [])
    moves = [item['title'] for item in raw_results]

    if moves:
        df_moves = pd.DataFrame(moves)
        
        # FIX: Replace empty 'name' strings with 'input' strings (e.g., 5P)
        df_moves['name'] = df_moves['name'].replace('', None)
        df_moves['name'] = df_moves['name'].fillna(df_moves['input'])
        
        # Mapping to your desired column format
        column_map = {
            "name": "Move",
            "damage": "Damage",
            "guard": "Guard",
            "startup": "Startup",
            "active": "Active",
            "recovery": "Recovery",
            "onBlock": "On-Block",
            "invuln": "Invuln"
        }
        df_moves = df_moves.rename(columns=column_map)
        
        # Clean up the extra 'input' column and sort for readability
        df_moves = df_moves.drop(columns=['input'])
        df_moves = df_moves.sort_values('Move')

        # Display the full table
        pd.set_option('display.max_rows', None)
        print(df_moves.to_string(index=False))
    else:
        print("No move data found in the database.")

except Exception as e:
    print(f"Error fetching data: {e}")

                      Move            Damage        Guard    Startup        Active               Recovery On-Block                                                         Invuln
                        2H                46          All         11             5                     31      -17                                                               
                        2K                28          Low          6             3                     11       -2                                                               
                        2P                22          All          5             4                      8       -2                                                               
                        2S                34          All         10             6                     15       -7                                                               
                        5H                52          All         11             4                     20     

In [4]:
import requests
import pandas as pd

# Cargo API endpoint for structured Dustloop data
api_url = "https://www.dustloop.com/wiki/api.php"

params = {
    "action": "cargoquery",
    "tables": "MoveData_GGST",
    "fields": "name, input, damage, guard, startup, active, recovery, onBlock, invuln",
    "where": 'chara="Sol Badguy"',
    "format": "json",
    "limit": "max" 
}

headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(api_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    raw_results = data.get('cargoquery', [])
    moves = [item['title'] for item in raw_results]

    if moves:
        df = pd.DataFrame(moves)
        
        # 1. FIX NAMES: Use 'input' if 'name' is empty
        df['name'] = df['name'].replace('', None)
        df['name'] = df['name'].fillna(df['input'])

        # 2. RENAME SPECIFIC MOVES: Sweep and Dust attacks
        name_map = {
            "Sweep": "2D",
            "Dust Attack": "5D",
            "Charged Dust Attack": "Charged 5D"
        }
        df['name'] = df['name'].replace(name_map)

        # 3. REMOVE UNWANTED MOVES: Wild Assault, Finish Blow, etc.
        exclude_moves = ["Wild Assault", "Charged Wild Assault", "Finish Blow"]
        df = df[~df['name'].isin(exclude_moves)]

        # 4. CLEAN RECOVERY COLUMN: Remove the word 'Total'
        df['recovery'] = df['recovery'].str.replace(r'(?i)Total', '', regex=True).str.strip()

        # 5. FILL BLANKS: Use 'n/a' for any remaining empty or null values
        df = df.replace(r'^\s*$', None, regex=True) # Convert whitespace-only to None
        df = df.fillna('n/a')

        # 6. FINAL FORMATTING: Rename columns and sort
        column_map = {
            "name": "Move",
            "damage": "Damage",
            "guard": "Guard",
            "startup": "Startup",
            "active": "Active",
            "recovery": "Recovery",
            "onBlock": "On-Block",
            "invuln": "Invuln"
        }
        df = df.rename(columns=column_map)
        
        if 'input' in df.columns:
            df = df.drop(columns=['input'])
        
        df = df.sort_values('Move')

        # 7. DISPLAY SETTINGS: Full table, no truncation
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)

        print(df.to_string(index=False))
    else:
        print("No move data found.")

except Exception as e:
    print(f"Error fetching data: {e}")

                      Move            Damage        Guard    Startup        Active               Recovery On-Block                                                         Invuln
                        2D                36          Low         10             3                     18       -4                         5-24 Low Profile<br>25-27 Below Crouch
                        2H                46          All         11             5                     31      -17                                                            n/a
                        2K                28          Low          6             3                     11       -2                                                            n/a
                        2P                22          All          5             4                      8       -2                                                            n/a
                        2S                34          All         10             6                     15     

In [3]:
import requests
import pandas as pd
import pymssql # Use pymssql for easy Docker connectivity

# --- 1. DATA FETCHING ---
api_url = "https://www.dustloop.com/wiki/api.php"
params = {
    "action": "cargoquery",
    "tables": "MoveData_GGST",
    "fields": "name, input, damage, guard, startup, active, recovery, onBlock, invuln",
    "where": 'chara="Sol Badguy"',
    "format": "json",
    "limit": "max" 
}
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(api_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    moves = [item['title'] for item in data.get('cargoquery', [])]

    if moves:
        df = pd.DataFrame(moves)
        
        # --- 2. CLEANUP & RENAMING ---
        df['name'] = df['name'].replace('', None).fillna(df['input'])
        name_map = {"Sweep": "2D", "Dust Attack": "5D", "Charged Dust Attack": "Charged 5D"}
        df['name'] = df['name'].replace(name_map)

        exclude = ["Wild Assault", "Charged Wild Assault", "Finish Blow"]
        df = df[~df['name'].isin(exclude)]

        df['recovery'] = df['recovery'].str.replace(r'(?i)Total', '', regex=True).str.strip()
        df = df.replace(r'^\s*$', None, regex=True).fillna('n/a')

        column_map = {
            "name": "Move", "damage": "Damage", "guard": "Guard",
            "startup": "Startup", "active": "Active", "recovery": "Recovery",
            "onBlock": "On-Block", "invuln": "Invuln"
        }
        df = df.rename(columns=column_map)
        if 'input' in df.columns: df = df.drop(columns=['input'])
        df = df.sort_values('Move')

        # --- 3. MSSQL EXPORT ---
        # Connection details for your Docker container
        server = 'localhost'
        user = 'sa' # Default MSSQL user
        password = '&^JAdS0Hc2aD2B'
        database = 'Dashloop' # Or create a specific DB first

        conn = pymssql.connect(server, user, password, database)
        
        # Using SQLAlchemy for the actual export (cleaner with Pandas)
        from sqlalchemy import create_engine
        engine = create_engine(f"mssql+pymssql://{user}:{password}@{server}/{database}")
        
        df.to_sql('SolBadguyFrameData', engine, if_exists='replace', index=False)
        conn.close()
        
        print("Data successfully exported to SQL Server (sql_server container)!")

        # --- 4. DISPLAY ---
        pd.set_option('display.max_rows', None)
        pd.set_option('display.width', None)
        print(df.to_string(index=False))
        
    else:
        print("No data found.")

except Exception as e:
    print(f"Error: {e}")

Data successfully exported to SQL Server (sql_server container)!
                      Move            Damage        Guard    Startup        Active               Recovery On-Block                                                         Invuln
                        2D                36          Low         10             3                     18       -4                         5-24 Low Profile<br>25-27 Below Crouch
                        2H                46          All         11             5                     31      -17                                                            n/a
                        2K                28          Low          6             3                     11       -2                                                            n/a
                        2P                22          All          5             4                      8       -2                                                            n/a
                        2S                34 